In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l8.1 KV caching

During generation, the keys and values of past tokens never change, yet naive
decoding recomputes them every step. A **KV cache** stores them, so each new token
is O(1) extra work instead of O(T). The output is identical, that parity is the
thing to prove.

In [ ]:
from pocketlm import scaled_dot_product_attention, KVCache, cached_step

torch.manual_seed(0)
B, H, T, hs = 1, 2, 6, 8
q = torch.randn(B, H, T, hs)
k = torch.randn(B, H, T, hs)
v = torch.randn(B, H, T, hs)
full, _ = scaled_dot_product_attention(q, k, v, causal=True)
cache = KVCache()
outs = [cached_step(q[:, :, t:t + 1], k[:, :, t:t + 1], v[:, :, t:t + 1], cache)
        for t in range(T)]
incremental = torch.cat(outs, dim=2)
print("cached == full attention:", torch.allclose(incremental, full, atol=1e-5))

Step-by-step cached attention reproduces full causal attention exactly. The cache
trades a little memory for a large compute saving, the core inference optimization.

In [ ]:
assert torch.allclose(incremental, full, atol=1e-5)